# Evolving a RAI Agent in One Workshop Generation

This notebook is the RAI counterpart to the toy RHO/CaP-X exercise. It does **not** reproduce the O3DE paper experiment or replay its results.

Instead, a real RAI tool-calling agent controls a tiny in-memory tabletop world. Its repository begins with two related defects: the prompt reverses the left/right coordinate convention, and `tools.py` removes the sign from requested coordinates. We then:

1. run the broken seed agent on one held-out object;
2. give HELIX/OpenCode one generation to edit `solver/prompt.py` and `solver/tools.py`;
3. display the selected diff and the complete evolved prompt and tools;
4. rerun the same held-out task and require a passing manipulation test.

The world is deterministic and simulator-free, so the exercise demonstrates repository evolution in minutes while still using RAI for the actual agent/tool loop.

## The toy repository-as-policy

The candidate is a normal Git repository:

```text
candidate/
  solver/
    prompt.py       # mutable RAI system prompt
    tools.py        # mutable LangChain tools used by RAI
  CONTRACT.md       # protected task and API contract
  scenarios.json    # protected train/validation/test IDs
  probe.py          # protected HELIX evaluator
  helix.toml        # protected evolution policy
  opencode.json     # protected edit permissions
```

The evaluator creates a fresh in-memory world for each task, builds the candidate's tools, and runs `rai.agents.langchain.create_conversational_agent`. HELIX sees a red-cube training task and a blue-cylinder validation task. The green-cube test remains outside evolution. Only files under `solver/` are editable.

In [ ]:
import hashlib
import json
import os
import sys
from pathlib import Path

from IPython.display import Markdown, display

sys.path.insert(0, "/ryzers/notebooks/scripts")

import rai_toy_demo as rai_demo

ROOT = Path("/tmp/rai_toy_notebook/candidate")
MODEL = "Gemma-4-E2B-it-GGUF"
GENERATIONS = 1
HELIX_TIMEOUT_SECONDS = 600
TASK_TIMEOUT_SECONDS = 120
TEST_TASK = "test-green-cube"

# Mock mode exists for CI plumbing only; the workshop must run the real RAI loop.
assert os.environ.get("RAI_TOY_MOCK") != "1", (
    "Unset RAI_TOY_MOCK before running this live RAI notebook."
)

print("RAI/OpenCode model:", MODEL)
print("Evolution budget:", GENERATIONS, "generation")
print("Held-out task:", TEST_TASK)

## 1. Load the model and build the broken seed

The same local Gemma E2B model plays two roles sequentially: RAI decides which tools to call during evaluation, and OpenCode edits the repository during mutation. Its GGUF is baked into `/opt/lemonade-cache`, so loading it does not trigger a network download. There is no O3DE, ROS graph, perception model, or GPU renderer to start.

`prepare_workshop()` creates fresh train, validation, and test scenarios. The test ID is protected and never included in HELIX batches.

In [ ]:
rai_demo.ensure_model(MODEL)
ROOT = rai_demo.prepare_workshop(
    ROOT,
    model=MODEL,
    generations=GENERATIONS,
)

manifest = json.loads((ROOT / "scenarios.json").read_text())
assert "test" not in manifest["splits"]
assert TEST_TASK not in manifest["scenarios"]
assert manifest["test_exposed_to_evolution"] is False

print("Candidate repository:", ROOT)
print("Train:", manifest["splits"]["train"])
print("Validation:", manifest["splits"]["val"])
print("Held-out test (not stored in candidate):", TEST_TASK)

## 2. Inspect the seed prompt and tools

The defect is intentionally small enough to understand during a workshop. The prompt says positive y is left, while the world says negative y is left. The movement wrapper then applies `abs()` to every requested y value, so even a correct negative target becomes positive.

The evaluator returns the actual RAI tool trace and final world state. OpenCode receives those diagnostics but cannot edit the evaluator or scenarios.

In [ ]:
SEED_PROMPT = (ROOT / "solver/prompt.py").read_text()
SEED_TOOLS = (ROOT / "solver/tools.py").read_text()
SEED_SHA256 = hashlib.sha256((SEED_PROMPT + SEED_TOOLS).encode()).hexdigest()

print("--- solver/prompt.py ---")
print(SEED_PROMPT)
print("--- relevant solver/tools.py line ---")
for line in SEED_TOOLS.splitlines():
    if "normalized_y" in line:
        print(line)
print("Seed policy SHA-256:", SEED_SHA256)

## 3. Run the broken seed on the held-out task

RAI receives: “Move the green cube into the left target centered at y=-0.50.” The toy world records every tool call independently of the model's prose. Because the seed movement tool folds `-0.50` to `+0.50`, this policy cannot pass.

This held-out result is shown to workshop participants, but it is not written into the candidate repository or exposed to HELIX.

In [ ]:
before = rai_demo.score_candidate(
    ROOT,
    TEST_TASK,
    timeout_seconds=TASK_TIMEOUT_SECONDS,
)
assert before["side_info"]["is_live_rai"] is True
assert before["passed"] is False, "The deliberately broken seed unexpectedly passed"
assert (ROOT / "solver/prompt.py").read_text() == SEED_PROMPT
assert (ROOT / "solver/tools.py").read_text() == SEED_TOOLS

display({
    "score": before["score"],
    "passed": before["passed"],
    "final_y": before["side_info"]["final_y"],
    "target_y": before["side_info"]["target_y"],
    "tool_trace": before["side_info"]["tool_trace"],
    "agent_response": before["side_info"]["agent_response"],
})

## 4. Run one HELIX/OpenCode generation

HELIX evaluates the seed on the red-cube training task, asks OpenCode to edit both solver files, and checks the child on the blue-cylinder validation task. Strict improvement decides whether the child enters the frontier.

The evaluator feedback contains the RAI tool trace and final coordinate. OpenCode can also read `CONTRACT.md`, which explains the required signed-coordinate behavior. The green-cube task remains held out.

In [ ]:
helix_run = rai_demo.run_helix(
    ROOT,
    generations=GENERATIONS,
    timeout_seconds=HELIX_TIMEOUT_SECONDS,
)
summary = rai_demo.summarize_run(ROOT)

print(
    f"HELIX exit={helix_run.returncode}  timed_out={helix_run.timed_out}  "
    f"elapsed={helix_run.elapsed_seconds:.1f}s"
)
print("Accepted improved repository:", summary["improved_best"])
print("Frontier:", summary["frontier"])

if helix_run.timed_out:
    raise TimeoutError("HELIX exceeded the workshop deadline")
if helix_run.returncode != 0:
    raise RuntimeError(helix_run.stdout[-6000:])
if not summary["improved_best"]:
    raise RuntimeError(
        "No improved child was selected. Inspect the HELIX output and rerun "
        "this one-generation cell before continuing."
    )

EVOLVED_ROOT = Path(summary["best"])
print("Selected repository:", EVOLVED_ROOT)

## 5. Show exactly what evolved

Unlike a model-weight update, repository evolution leaves a reviewable software artifact. The next cell prints the unified diff followed by the complete selected `prompt.py` and `tools.py`, so participants can inspect both the language-level and implementation-level repair.

In [ ]:
EVOLVED_PROMPT = (EVOLVED_ROOT / "solver/prompt.py").read_text()
EVOLVED_TOOLS = (EVOLVED_ROOT / "solver/tools.py").read_text()
EVOLVED_SHA256 = hashlib.sha256(
    (EVOLVED_PROMPT + EVOLVED_TOOLS).encode()
).hexdigest()

print("--- accepted source diff ---")
print(summary["diff"])
print("\n--- evolved solver/prompt.py ---")
print(EVOLVED_PROMPT)
print("\n--- evolved solver/tools.py ---")
print(EVOLVED_TOOLS)
print("Evolved policy SHA-256:", EVOLVED_SHA256)

## 6. Rerun the same held-out manipulation test

The selected repository now controls a fresh RAI agent in a freshly reset world. The task, green object, initial coordinate, target coordinate, model, and evaluator are unchanged. Only the repository differs.

A pass requires all of the following: the evolved prompt states the correct convention, RAI observes before moving, the movement tool preserves a negative target, and the green cube finishes at y=-0.50.

In [ ]:
after = rai_demo.score_candidate(
    EVOLVED_ROOT,
    TEST_TASK,
    timeout_seconds=TASK_TIMEOUT_SECONDS,
)
assert after["side_info"]["is_live_rai"] is True

comparison = {
    "seed_score": before["score"],
    "evolved_score": after["score"],
    "seed_passed": before["passed"],
    "evolved_passed": after["passed"],
    "target_y": after["side_info"]["target_y"],
    "final_y": after["side_info"]["final_y"],
    "tool_trace": after["side_info"]["tool_trace"],
    "agent_response": after["side_info"]["agent_response"],
}
display(comparison)

if not after["passed"]:
    raise RuntimeError(
        "The selected repository did not pass the held-out RAI task. Rerun the "
        "one-generation evolution cell to obtain another live mutation."
    )
print("PASS: evolved RAI policy moved the held-out green cube to the left target.")

In [ ]:
report = {
    "schema_version": "rai-toy-repository-evolution/v1",
    "benchmark": "rai_toy_in_memory_manipulation",
    "is_live_rai": True,
    "model": MODEL,
    "generations": GENERATIONS,
    "splits": manifest["splits"],
    "held_out_test": TEST_TASK,
    "test_exposed_to_evolution": False,
    "seed_policy_sha256": SEED_SHA256,
    "evolved_policy_sha256": EVOLVED_SHA256,
    "accepted_diff": summary["diff"],
    "before": before,
    "after": after,
}
report_path = ROOT.parent / "rai_toy_one_generation_report.json"
report_path.write_text(json.dumps(report, indent=2) + "\n")
print("Wrote", report_path)

In [ ]:
display(Markdown(f"""
## Live result

- Seed held-out test: **{'PASS' if before['passed'] else 'FAIL'}** (score {before['score']:.2f})
- Evolved held-out test: **{'PASS' if after['passed'] else 'FAIL'}** (score {after['score']:.2f})
- HELIX generations: **{GENERATIONS}**
- Evolution wall time: **{helix_run.elapsed_seconds:.1f} seconds**
- Changed policy: **{SEED_SHA256 != EVOLVED_SHA256}**

This is a live toy RAI result from the current workshop run—not a reproduced or prerecorded O3DE result. The selected prompt and tools printed above are the exact files used by the passing test.
"""))

## What this demonstrates

RAI remains the runtime agent: it interprets the natural-language task and decides when and how to invoke candidate tools. HELIX does not update model weights; it selects a new versioned prompt-and-tools repository from evaluator feedback.

The toy boundary is intentionally replaceable. A larger project can keep the same structure while swapping `ToyWorld` for ROS 2, O3DE, or hardware:

1. keep prompts and tools in a narrow mutable directory;
2. keep task reset, scoring, and split definitions protected;
3. return traces and errors that help mutation without leaking test answers;
4. freeze the selected repository before final deployment tests.

The implementation used here is `scripts/rai_toy_demo.py`. Its static contract test is `tests/test_rai_toy_evolution.sh`.

In [ ]:
print("Candidate repository:", ROOT)
print("Selected repository:", EVOLVED_ROOT)
print("Run report:", report_path)
print("Lemonade remains loaded for the next workshop notebook.")